# MQAR Zoology Benchmark: SKA-Mamba vs Mamba vs Mamba+Attention

Tests multi-query associative recall (MQAR) from the Zoology paper (Arora et al., ICLR 2024).

Three models compared at matched parameter count:
1. **Mamba-only**: pure Mamba-2 blocks + FFN
2. **Mamba+Attention**: Mamba blocks with one attention layer
3. **SKA-Mamba**: Mamba-2 + SKA spectral Koopman blocks (our method)

**Runtime**: Set to GPU (Runtime > Change runtime type > T4 or better)

## Install dependencies

In [ ]:
!pip install torch --quiet

## MQAR data generation

In [ ]:
import math
import time
import random
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")


def generate_mqar_data(
    num_examples: int,
    seq_len: int,
    num_kv_pairs: int,
    vocab_size: int = 8192,
    seed: int = 42,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Generate MQAR data following Zoology conventions.

    Sequence layout:
      [k1, v1, k2, v2, ..., k_P, v_P, noise..., q1, q2, ..., q_P]

    Keys are drawn from [2, vocab_size) to reserve 0=pad, 1=query_marker.
    Values are drawn from [2, vocab_size).
    Queries repeat a key, and the label at that position is the paired value.
    Non-query positions have label 0 (ignored in loss).

    Returns:
      inputs: (num_examples, seq_len) long tensor
      labels: (num_examples, seq_len) long tensor, 0 at non-query positions
    """
    torch.manual_seed(seed)

    inputs = torch.zeros(num_examples, seq_len, dtype=torch.long)
    labels = torch.zeros(num_examples, seq_len, dtype=torch.long)

    # Fill with noise tokens
    inputs[:] = torch.randint(2, vocab_size, (num_examples, seq_len))

    kv_region = 2 * num_kv_pairs
    query_region_start = seq_len - num_kv_pairs

    for i in range(num_examples):
        # Sample unique keys
        keys = torch.randperm(vocab_size - 2)[:num_kv_pairs] + 2
        values = torch.randint(2, vocab_size, (num_kv_pairs,))

        # Place key-value pairs at the start
        for p in range(num_kv_pairs):
            inputs[i, 2 * p] = keys[p]
            inputs[i, 2 * p + 1] = values[p]

        # Place queries at the end, record labels
        perm = torch.randperm(num_kv_pairs)
        for j in range(num_kv_pairs):
            q_pos = query_region_start + j
            kv_idx = perm[j]
            inputs[i, q_pos] = keys[kv_idx]
            labels[i, q_pos] = values[kv_idx]

    return inputs, labels

Device: cuda
GPU: NVIDIA H100 80GB HBM3


## shared building blocks

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).to(x.dtype) * self.weight


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, d_ffn: int):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.w_gate = nn.Linear(d_model, d_ffn, bias=False)
        self.w_up = nn.Linear(d_model, d_ffn, bias=False)
        self.w_down = nn.Linear(d_ffn, d_model, bias=False)

    def forward(self, x):
        h = self.norm(x)
        return x + self.w_down(F.silu(self.w_gate(h)) * self.w_up(h))

## Mamba-2 block (simplified, self-contained, no external deps)

In [ ]:
class SimpleMamba2(nn.Module):
    """
    Simplified Mamba-2 SSD block for benchmarking.
    Uses chunked parallel scan with complex state.
    """

    def __init__(self, d_model: int, d_state: int = 64, d_conv: int = 4,
                 expand: int = 2, n_heads: int = 1, chunk_size: int = 64):
        super().__init__()
        self.d_model = d_model
        self.d_inner = d_model * expand
        self.n_heads = n_heads
        self.d_state = d_state
        self.d_head = self.d_inner // n_heads
        self.chunk_size = chunk_size

        self.norm = RMSNorm(d_model)

        # SSM projections
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner + 2 * n_heads * d_state + 3 * n_heads, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True)

        self.dt_bias = nn.Parameter(torch.zeros(n_heads))
        self.a_bias = nn.Parameter(torch.log(torch.linspace(0.9, 0.999, n_heads)))

    def forward(self, x):
        residual = x
        x = self.norm(x)
        B, T, D = x.shape
        H = self.n_heads
        N = self.d_state
        P = self.d_head

        proj = self.in_proj(x)
        z, x_in, B_proj, C_proj, dt_raw, a_raw, d_raw = proj.split(
            [self.d_inner, self.d_inner, H * N, H * N, H, H, H], dim=-1)

        # Conv + SiLU on x
        x_conv = self.conv1d(x_in.transpose(1, 2))[..., :T].transpose(1, 2)
        x_act = F.silu(x_conv).reshape(B, T, H, P)

        # Gate
        z_act = F.silu(z).reshape(B, T, H, P)

        # Discretize
        dt = F.softplus(dt_raw + self.dt_bias).clamp(1e-4, 1.0)
        a = F.softplus(a_raw + self.a_bias)
        alpha = torch.exp(-a * dt).unsqueeze(-1)

        # B and C projections
        B_proj = B_proj.reshape(B, T, H, N)
        C_proj = C_proj.reshape(B, T, H, N)

        # Simple recurrent scan (sequential for correctness, GPU-accelerated via tensor ops)
        dt_exp = dt.unsqueeze(-1)
        h = torch.zeros(B, H, N, P, device=x.device, dtype=torch.float32)
        outputs = []

        for t in range(T):
            b_t = B_proj[:, t]
            c_t = C_proj[:, t]
            x_t = x_act[:, t].float()
            a_t = alpha[:, t]

            h = a_t.unsqueeze(-1) * h + torch.einsum("bhn,bhp->bhnp", b_t.float() * dt_exp[:, t], x_t)
            y_t = torch.einsum("bhn,bhnp->bhp", c_t.float(), h)
            outputs.append(y_t)

        y = torch.stack(outputs, dim=1).to(x.dtype)

        # Gate and project
        y = (y * z_act).reshape(B, T, self.d_inner)
        return residual + self.out_proj(y)

## Causal multi-head attention block

In [ ]:
class CausalMHA(nn.Module):
    """Standard causal multi-head attention for the hybrid baseline."""

    def __init__(self, d_model: int, n_heads: int = 4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.norm = RMSNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        h = self.norm(x)
        q, k, v = self.qkv(h).chunk(3, dim=-1)
        q = q.reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product with causal mask (uses PyTorch SDPA for GPU efficiency)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).reshape(B, T, D)
        return x + self.out_proj(y)

## SKA block (Koopman spectral retrieval, numerically stabilized)

In [ ]:
class SKABlock(nn.Module):
    """
    Spectral Koopman Attention block.

    Builds Gram matrix G = Z^T Z + ridge*I from projected keys,
    eigendecomposes for stable solve, power-filters queries through
    the Koopman operator, retrieves values. O(T * r^2) complexity,
    O(r^2) state size.
    """

    def __init__(self, d_model: int, n_heads: int = 4, rank: int = 32,
                 ridge: float = 1e-2, power_K: int = 2, scale: float = 1.5):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.rank = rank
        self.ridge = ridge
        self.power_K = power_K

        self.norm = RMSNorm(d_model)
        self.key_proj = nn.Linear(d_model, n_heads * rank, bias=False)
        self.query_proj = nn.Linear(d_model, n_heads * rank, bias=False)
        self.value_proj = nn.Linear(d_model, n_heads * d_model // n_heads, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        nn.init.orthogonal_(self.key_proj.weight)
        nn.init.orthogonal_(self.query_proj.weight)

        self.eta = nn.Parameter(torch.tensor(scale))
        self.ssn_gamma = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        B, T, D = x.shape
        H, r, P = self.n_heads, self.rank, self.d_head

        h = self.norm(x)
        z = self.key_proj(h).reshape(B, T, H, r)
        zq = self.query_proj(h).reshape(B, T, H, r)
        v = self.value_proj(h).reshape(B, T, H, P)

        # Compute in float32
        z_f = z.float()
        zq_f = zq.float()
        v_f = v.float()

        # Sequence max normalization
        z_norms = z_f.norm(dim=-1, keepdim=True)
        max_norm = z_norms.max(dim=1, keepdim=True)[0].clamp(min=1e-6)
        z_f = z_f / max_norm
        zq_f = zq_f / max_norm

        # Gram matrix: G = Z^T Z + ridge * I
        # z_f: (B, T, H, r) -> need (B, H, r, T) for matmul
        z_perm = z_f.permute(0, 2, 3, 1)
        G = z_perm @ z_perm.transpose(-1, -2)
        G = G + self.ridge * torch.eye(r, device=x.device, dtype=torch.float32)

        # Cross-covariance: M = Z[1:]^T Z[:-1]
        z_next = z_f[:, 1:].permute(0, 2, 3, 1)
        z_prev = z_f[:, :-1].permute(0, 2, 3, 1)
        M = z_next @ z_prev.transpose(-1, -2)

        # Value cross-covariance: C_v = V^T Z
        C_v = torch.einsum("bthp,bthr->bhpr", v_f, z_f)

        # Eigendecomposition of G
        G_sym = 0.5 * (G + G.transpose(-1, -2))
        eigvals, eigvecs = torch.linalg.eigh(G_sym)
        eigvals = eigvals.clamp(min=1e-6)
        inv_eigvals = 1.0 / eigvals

        # G^{-1} via eigen: G_inv @ X = U @ diag(1/lambda) @ U^T @ X
        def solve(X):
            Ut_X = eigvecs.transpose(-1, -2) @ X
            n_trail = Ut_X.dim() - inv_eigvals.dim()
            ie = inv_eigvals
            for _ in range(n_trail):
                ie = ie.unsqueeze(-1)
            return eigvecs @ (ie * Ut_X)

        # Koopman operator: A_w = G^{-1} M
        A_w = solve(M)

        # Spectral normalization via power iteration
        v_pi = torch.randn(B, H, r, 1, device=x.device, dtype=torch.float32)
        v_pi = v_pi / v_pi.norm(dim=-2, keepdim=True).clamp(min=1e-8)
        for _ in range(6):
            u_pi = A_w @ v_pi
            u_pi = u_pi / u_pi.norm(dim=-2, keepdim=True).clamp(min=1e-8)
            v_pi = A_w.transpose(-1, -2) @ u_pi
            v_pi = v_pi / v_pi.norm(dim=-2, keepdim=True).clamp(min=1e-8)
        sigma = (A_w @ v_pi).norm(dim=-2).squeeze(-1)
        scale_sn = sigma.clamp(min=1.0).unsqueeze(-1).unsqueeze(-1)
        A_w = A_w / scale_sn
        gamma = torch.clamp(self.ssn_gamma, min=1.0, max=1.5)
        A_w = A_w * gamma

        # Value operator: B_v = G^{-1} C_v^T
        B_v = solve(C_v.transpose(-1, -2)).transpose(-1, -2)

        # G^{-1/2} and G^{1/2} for whitening/coloring
        isqrt_eig = (1.0 / eigvals.clamp(min=1e-6).sqrt()).unsqueeze(-2)
        G_inv_sqrt = (eigvecs * isqrt_eig) @ eigvecs.transpose(-1, -2)
        sqrt_eig = eigvals.sqrt().unsqueeze(-2)
        G_sqrt = (eigvecs * sqrt_eig) @ eigvecs.transpose(-1, -2)

        # Power-filtered spectral query
        zq_perm = zq_f.permute(0, 2, 3, 1)
        w_q = G_inv_sqrt @ zq_perm
        w_f = w_q
        for _ in range(self.power_K):
            w_f = A_w @ w_f
        z_filtered = G_sqrt @ w_f

        # Retrieve values
        y_hat = (B_v @ z_filtered).permute(0, 3, 1, 2)

        y_out = self.eta * y_hat.to(x.dtype)
        y_out = self.out_proj(y_out.reshape(B, T, H * P))
        return x + y_out

## Three model architectures

In [ ]:
class MambaOnlyModel(nn.Module):
    """Baseline: pure Mamba-2 blocks + FFN."""

    def __init__(self, vocab_size, d_model, n_layers, d_state=64, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        d_ffn = ((int(d_model * 2.667) + 63) // 64) * 64

        layers = []
        for _ in range(n_layers):
            layers.append(SimpleMamba2(d_model, d_state=d_state, n_heads=n_heads))
            layers.append(SwiGLUFFN(d_model, d_ffn))
        self.layers = nn.ModuleList(layers)
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h = layer(h)
        return self.head(self.norm_f(h))


class MambaAttentionModel(nn.Module):
    """Hybrid: Mamba blocks with one attention layer replacing the last Mamba."""

    def __init__(self, vocab_size, d_model, n_layers, d_state=64, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        d_ffn = ((int(d_model * 2.667) + 63) // 64) * 64

        layers = []
        for i in range(n_layers):
            if i == n_layers - 1:
                # Last layer uses attention instead of Mamba
                layers.append(CausalMHA(d_model, n_heads=n_heads))
            else:
                layers.append(SimpleMamba2(d_model, d_state=d_state, n_heads=n_heads))
            layers.append(SwiGLUFFN(d_model, d_ffn))
        self.layers = nn.ModuleList(layers)
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h = layer(h)
        return self.head(self.norm_f(h))


class SKAMambaModel(nn.Module):
    """Our method: each block = Mamba + SKA + FFN."""

    def __init__(self, vocab_size, d_model, n_layers, d_state=64,
                 n_heads=4, ska_rank=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        d_ffn = ((int(d_model * 2.667) + 63) // 64) * 64

        layers = []
        for _ in range(n_layers):
            layers.append(SimpleMamba2(d_model, d_state=d_state, n_heads=n_heads))
            layers.append(SKABlock(d_model, n_heads=n_heads, rank=ska_rank))
            layers.append(SwiGLUFFN(d_model, d_ffn))
        self.layers = nn.ModuleList(layers)
        self.norm_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h = layer(h)
        return self.head(self.norm_f(h))

## Training and evaluation

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


def train_and_eval(
    model,
    train_inputs,
    train_labels,
    test_inputs,
    test_labels,
    epochs: int = 60,
    batch_size: int = 64,
    lr: float = 3e-4,
    label: str = "model",
):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Warmup + cosine schedule
    total_steps = epochs * (len(train_inputs) // batch_size + 1)
    warmup_steps = total_steps // 10

    def lr_fn(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)

    train_ds = TensorDataset(train_inputs, train_labels)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              pin_memory=True, drop_last=True)

    # Query mask for loss computation
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_acc = 0.0
    t0 = time.time()
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        n_batches = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            query_mask = (yb > 0)

            optimizer.zero_grad(set_to_none=True)

            if scaler:
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(xb)
                    loss_all = F.cross_entropy(
                        logits.view(-1, logits.size(-1)),
                        yb.view(-1),
                        reduction="none",
                    ).view_as(yb)
                    loss = (loss_all * query_mask).sum() / query_mask.sum().clamp(min=1)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(xb)
                loss_all = F.cross_entropy(
                    logits.view(-1, logits.size(-1)),
                    yb.view(-1),
                    reduction="none",
                ).view_as(yb)
                loss = (loss_all * query_mask).sum() / query_mask.sum().clamp(min=1)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            scheduler.step()
            total_loss += loss.item()
            n_batches += 1

        # Evaluate
        model.eval()
        with torch.no_grad():
            test_x = test_inputs.to(device)
            test_y = test_labels.to(device)

            if scaler:
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    test_logits = model(test_x)
            else:
                test_logits = model(test_x)

            test_preds = test_logits.argmax(dim=-1)
            query_mask_test = (test_y > 0)
            correct = ((test_preds == test_y) & query_mask_test).sum().item()
            total = query_mask_test.sum().item()
            acc = correct / max(total, 1)
            best_acc = max(best_acc, acc)

        avg_loss = total_loss / max(n_batches, 1)
        history.append({"epoch": epoch, "loss": avg_loss, "acc": acc})

        if (epoch + 1) % 10 == 0 or epoch == 0:
            elapsed = time.time() - t0
            print(f"  [{label}] epoch {epoch+1:3d}/{epochs} | "
                  f"loss {avg_loss:.4f} | acc {acc:.4f} | "
                  f"best {best_acc:.4f} | {elapsed:.1f}s")

    return best_acc, history

## Run the benchmark

In [ ]:
def run_mqar_benchmark():
    print("=" * 70)
    print("MQAR Zoology Benchmark: SKA-Mamba vs Mamba vs Mamba+Attention")
    print("=" * 70)

    # Benchmark configurations
    # Following Zoology: vocab=8192, 2-layer models, sweep sequence lengths
    configs = [
        {"seq_len": 64,  "num_kv_pairs": 4,  "d_model": 64,  "n_layers": 2},
        {"seq_len": 128, "num_kv_pairs": 8,  "d_model": 64,  "n_layers": 2},
        {"seq_len": 256, "num_kv_pairs": 16, "d_model": 64,  "n_layers": 2},
        {"seq_len": 512, "num_kv_pairs": 32, "d_model": 64,  "n_layers": 2},
    ]

    vocab_size = 8192
    num_train = 10000
    num_test = 1000
    epochs = 80
    batch_size = 64
    lr = 1e-3

    all_results = {}

    for cfg in configs:
        seq_len = cfg["seq_len"]
        num_kv = cfg["num_kv_pairs"]
        d_model = cfg["d_model"]
        n_layers = cfg["n_layers"]

        print(f"\n{'='*70}")
        print(f"Config: seq_len={seq_len}, num_kv_pairs={num_kv}, "
              f"d_model={d_model}, n_layers={n_layers}")
        print(f"{'='*70}")

        # Generate data
        train_x, train_y = generate_mqar_data(num_train, seq_len, num_kv, vocab_size)
        test_x, test_y = generate_mqar_data(num_test, seq_len, num_kv, vocab_size, seed=999)

        # Model configs
        n_heads = max(1, d_model // 16)
        d_state = 32
        ska_rank = max(16, d_model // 2)

        models = {
            "Mamba-only": MambaOnlyModel(
                vocab_size, d_model, n_layers, d_state=d_state, n_heads=n_heads),
            "Mamba+Attn": MambaAttentionModel(
                vocab_size, d_model, n_layers, d_state=d_state, n_heads=n_heads),
            "SKA-Mamba": SKAMambaModel(
                vocab_size, d_model, n_layers, d_state=d_state,
                n_heads=n_heads, ska_rank=ska_rank),
        }

        results_at_len = {}
        for name, model in models.items():
            n_params = count_params(model)
            print(f"\n  {name}: {n_params:,} params")
            best_acc, history = train_and_eval(
                model, train_x, train_y, test_x, test_y,
                epochs=epochs, batch_size=batch_size, lr=lr, label=name)
            results_at_len[name] = {"best_acc": best_acc, "params": n_params}

            # Free GPU memory
            del model
            torch.cuda.empty_cache() if device.type == "cuda" else None

        all_results[seq_len] = results_at_len

    # Print summary table
    print(f"\n\n{'='*70}")
    print("SUMMARY: Best accuracy per model and sequence length")
    print(f"{'='*70}")
    print(f"{'Seq Len':>10s} | {'Mamba-only':>12s} | {'Mamba+Attn':>12s} | {'SKA-Mamba':>12s}")
    print("-" * 55)
    for seq_len in sorted(all_results.keys()):
        r = all_results[seq_len]
        mamba_acc = r["Mamba-only"]["best_acc"]
        attn_acc = r["Mamba+Attn"]["best_acc"]
        ska_acc = r["SKA-Mamba"]["best_acc"]
        best = max(mamba_acc, attn_acc, ska_acc)
        line = f"{seq_len:>10d} | {mamba_acc:>11.4f} | {attn_acc:>11.4f} | {ska_acc:>11.4f}"
        # Mark the winner
        if ska_acc == best:
            line += "  *"
        elif attn_acc == best:
            line += "  (attn)"
        print(line)

    print(f"\nParam counts (seq_len={configs[0]['seq_len']}):")
    r0 = all_results[configs[0]["seq_len"]]
    for name, data in r0.items():
        print(f"  {name}: {data['params']:,}")

    return all_results

## Run it

In [ ]:
if __name__ == "__main__":
    results = run_mqar_benchmark()

MQAR Zoology Benchmark: SKA-Mamba vs Mamba vs Mamba+Attention

Config: seq_len=64, num_kv_pairs=4, d_model=64, n_layers=2

  Mamba-only: 1,207,376 params
